# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access the metadata object
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}, Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs as described by the Croissant schema. All references are by `@id`.

In [ ]:
# List all record sets and their fields
record_sets = []

if hasattr(metadata, 'record_sets'):
    print('Record sets and fields:')
    for recset in metadata.record_sets:
        print(f"- RecordSet @id: {recset['@id']}, name: {recset.get('name','')}")
        fields = recset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field['@id']}, name: {field.get('name','')}")
            else:
                print(f"    - Field @id: {field}")
        record_sets.append(recset['@id'])
else:
    # Fallback in case metadata.record_sets is not available directly
    print('No record sets defined under the record_sets property.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as listed above.

In [ ]:
# Here we collect all available record sets by their @id
if not record_sets:
    # If previous cell didn't find them, try to extract from metadata.to_json()
    js = metadata.to_json()
    if 'recordSet' in js:
        rs = js['recordSet']
        if isinstance(rs, list):
            record_sets = [r['@id'] if isinstance(r, dict) else r for r in rs]
        elif isinstance(rs, dict) and '@id' in rs:
            record_sets = [rs['@id']]
print('Available record_sets:', record_sets)

dataframes = {}

for record_set in record_sets:
    print(f'-> Loading records from RecordSet {record_set}')
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        print(f"Fields (columns) for record set {record_set}:\n", df.columns.tolist(), "\nFirst 3 records:")
        print(df.head(3))
        dataframes[record_set] = df
    else:
        print(f"No records found in {record_set}.")

# For illustration, if there are record_sets, pick the first one for further analysis
if record_sets:
    main_record_set = record_sets[0]
else:
    main_record_set = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. All fields are referenced by their `@id`.

In [ ]:
# Choose a numeric field for EDA; substitute with a relevant @id from your inspection above
numeric_field = None
group_field = None

if main_record_set:
    df = dataframes[main_record_set]
    # Heuristic: pick the first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    # Heuristic: pick the first non-numeric column for grouping
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break

    if numeric_field:
        print(f"Using numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df.loc[:, f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f"mean_{numeric_field}")
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found to perform EDA. Please review your record set fields.")
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we display a histogram for the chosen numeric field and boxenplot by the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set and numeric_field:
    df = dataframes[main_record_set]
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxenplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
In this notebook, we loaded, inspected, and performed basic exploratory data analysis (EDA) on the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We referenced all schema entities by their `@id`, explored available record sets and fields, performed filtering and normalization, and visualized data distributions.

**Next steps:** Further domain-specific analysis can be done on the protein abundance and modification fields, or the data can be prepared for downstream bioinformatics or machine learning pipelines. Consult the dataset's [metadata and schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for detailed field definitions and usage.